In [1]:
import pandas as pd
import anthropic


In [2]:
source_data = pd.read_json('pubmedqa/data/pqaa_train_set.json').T

source_data['id'] = source_data.index.astype(int)
train = source_data.iloc[0:700]
test = source_data.iloc[700:1000]
train.head()

,QUESTION,CONTEXTS,LABELS,MESHES,YEAR,reasoning_required_pred,reasoning_free_pred,final_decision,LONG_ANSWER,id
22117569,Is an advance care planning model feasible in ...,[An effective advance care planning programme ...,"[BACKGROUND, METHODS, SETTINGS AND PARTICIPANT...","[Adult, Advance Care Planning, Advance Directi...",2012,yes,yes,yes,An advance care planning model is feasible for...,22117569
16097998,Is coeliac disease screening in risk groups ju...,[The benefits of serologic screening for coeli...,"[BACKGROUND, METHODS, RESULTS]","[Adult, Aged, Body Mass Index, Bone Density, C...",2005,yes,yes,yes,Long-term dietary compliance in screen-detecte...,16097998
15223725,Does blood pressure change in treated hyperten...,[To determine whether there are differences be...,"[OBJECTIVES, METHOD, RESULTS]","[Adult, Aged, Aged, 80 and over, Antihypertens...",2004,maybe,yes,yes,Systolic BP measured by the nurse in treated h...,15223725
17274051,Metastatic carcinoma to the cervical nodes fro...,[The aim of the study was to evaluate the outc...,"[BACKGROUND, METHODS, RESULTS]","[Biopsy, Fine-Needle, Carcinoma, Female, Head ...",2007,no,no,no,"In our experience, definitive radiotherapy to ...",17274051
27581329,Estimation of basal metabolic rate in Chinese:...,[Measurement of basal metabolic rate (BMR) is ...,"[BACKGROUND, METHODS, RESULTS]","[Adult, Aged, Asian Continental Ancestry Group...",2016,yes,yes,yes,"To date, the newly developed Singapore equatio...",27581329


In [ ]:
# how many sentences are in the long answers?
train['LONG_ANSWER'].apply(lambda x: len(x.split('.'))).describe()

count    700.000000
mean       2.962857
std        0.897488
min        1.000000
25%        2.000000
50%        3.000000
75%        3.000000
max        7.000000
Name: LONG_ANSWER, dtype: float64

<h1>Batch API</h1>

In [4]:
questions =source_data['QUESTION'].values.tolist()
ids = source_data['id'].values.tolist()
ids_and_questions = list(zip(ids, questions))

In [28]:
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

client = anthropic.Anthropic()

In [ ]:
# test message creation
question = 'Does coronary angiography before emergency aortic surgery affect in-hospital mortality?'
message = client.messages.create(
    model="claude-sonnet-5-5",
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content":"Answer the following medical question "
                                        "in 5 sentences or fewer.\n\n"
                                        f"{question}",
        }
    ],
)

for block in message.content:
    if block.type == "text":
        print(block.text)

Evidence suggests that routine coronary angiography before emergency aortic surgery (e.g., acute type A aortic dissection) does not improve in-hospital mortality and may worsen outcomes by delaying surgery. Several observational studies, including analyses from the International Registry of Acute Aortic Dissection (IRAD), found that preoperative angiography was not associated with a survival benefit, and in some cohorts it was linked to higher mortality because of delays and procedural risks such as dissection extension or malperfusion. Time to surgery is a critical determinant of survival in these patients, so CT angiography, which can assess coronary involvement and the aorta together, is generally preferred. Angiography may be reasonable selectively, such as in stable patients with known coronary disease or a history of prior revascularization, where results would change surgical planning. These conclusions come largely from nonrandomized data, so they are subject to selection bias.

<h1>Create Requests</h1>

In [ ]:
requests = []

for qid, question in ids_and_questions:
    requests.append(
        Request(
            custom_id=str(qid),
            params=MessageCreateParamsNonStreaming(
                model="claude-sonnet-5-5",
                max_tokens=1024,
                messages=[
                    {
                        "role": "user",
                        "content": (
                            "Answer the following medical question "
                            "in 3 sentences or fewer.\n\n"
                            f"{question}"
                        )
                    }
                ]
            )
        )
    )

<h1>Launch Batch</h1>

In [46]:
message_batch = client.messages.batches.create(
    requests=requests
)

print(message_batch.id)

msgbatch_01DiR8crZzt1qHQAEPybHmGg


<h1>Monitor Batch</h1>

In [50]:
batch = client.messages.batches.retrieve(
    message_batch.id
)

print(batch.processing_status)

ended


<h1>Get Results</h1>

In [60]:
results = client.messages.batches.results(
    message_batch.id
)
outputs = []

for result in results:
    if result.result.type == "succeeded":

        output = result.result.message.content#[0].text
        for block in output:
            if isinstance(block, anthropic.types.text_block.TextBlock):
                answer = block.text


        outputs.append({
            "id": result.custom_id,
            "answer": answer
        })

    else:
        outputs.append({
            "id": result.custom_id,
            "answer": None
        })

answers_df = pd.DataFrame(outputs)
answers_df.head(20)

,id,answer
0,22349635,Evidence suggests that selected elderly patien...
1,17276801,Yes. Elevated troponin I or T in acute pulmona...
2,14697414,"Yes. Oligometastatic prostate cancer, usually ..."
3,17621202,Current evidence suggests that shaving the inc...
4,16432652,Transgastric endoscopic splenectomy has been s...
5,25150098,"Yes, with some caveats. In black African popul..."
6,12855939,"Yes, in this population a low ankle-brachial i..."
7,8916748,"Yes, socioeconomic differences in mortality ge..."
8,18719011,"No, but the risk is substantially higher. Roug..."
9,9745063,Laparoscopic ultrasonography (LUS) is a useful...


<h1>Merge Answers DataFrame</h1>

In [ ]:
answers_df.rename(columns = {"answer": "SONNET 5-5 ANSWER"}, inplace=True)
answers_df['id'] = answers_df['id'].astype(int)
answers_df.head()


/tmp/ipykernel_22188/2709704613.py:1: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  answers_df.rename(columns = {"answer": "SONNET 5-5 ANSWER"}, inplace=True)


,id,SONNET 5-5 ANSWER
0,22349635,Evidence suggests that selected elderly patien...
1,17276801,Yes. Elevated troponin I or T in acute pulmona...
2,14697414,"Yes. Oligometastatic prostate cancer, usually ..."
3,17621202,Current evidence suggests that shaving the inc...
4,16432652,Transgastric endoscopic splenectomy has been s...


In [ ]:
final_df = source_data.merge(
    answers_df,
    left_on="id",
    right_on="id",
    how="left"
)
final_df.set_index('id', inplace=True)
final_df

,QUESTION,CONTEXTS,LABELS,MESHES,YEAR,reasoning_required_pred,reasoning_free_pred,final_decision,LONG_ANSWER,id,SONNET 5-5 ANSWER
0,Is an advance care planning model feasible in ...,[An effective advance care planning programme ...,"[BACKGROUND, METHODS, SETTINGS AND PARTICIPANT...","[Adult, Advance Care Planning, Advance Directi...",2012,yes,yes,yes,An advance care planning model is feasible for...,22117569,"Yes, evidence suggests that structured advance..."
1,Is coeliac disease screening in risk groups ju...,[The benefits of serologic screening for coeli...,"[BACKGROUND, METHODS, RESULTS]","[Adult, Aged, Body Mass Index, Bone Density, C...",2005,yes,yes,yes,Long-term dietary compliance in screen-detecte...,16097998,Targeted screening in at-risk groups is genera...
2,Does blood pressure change in treated hyperten...,[To determine whether there are differences be...,"[OBJECTIVES, METHOD, RESULTS]","[Adult, Aged, Aged, 80 and over, Antihypertens...",2004,maybe,yes,yes,Systolic BP measured by the nurse in treated h...,15223725,"Yes, in many treated hypertensive patients, bl..."
3,Metastatic carcinoma to the cervical nodes fro...,[The aim of the study was to evaluate the outc...,"[BACKGROUND, METHODS, RESULTS]","[Biopsy, Fine-Needle, Carcinoma, Female, Head ...",2007,no,no,no,"In our experience, definitive radiotherapy to ...",17274051,"Yes, in most patients neck dissection is neede..."
4,Estimation of basal metabolic rate in Chinese:...,[Measurement of basal metabolic rate (BMR) is ...,"[BACKGROUND, METHODS, RESULTS]","[Adult, Aged, Asian Continental Ancestry Group...",2016,yes,yes,yes,"To date, the newly developed Singapore equatio...",27581329,"Current prediction equations, such as the Harr..."
5,Recovery Outcome Measures: Is There a Place fo...,[Utilization of the Recovery Knowledge Invento...,[PURPOSE],"[Australia, Cultural Competency, Health Knowle...",2015,yes,yes,yes,This paper discusses the cultural appropriaten...,24964725,"Yes, culture, attitudes, and faith belong in r..."
6,Thoracic and thoracoabdominal aneurysm repair:...,[The impact of different strategies for manage...,"[BACKGROUND, METHODS, RESULTS]","[Adult, Aged, Aged, 80 and over, Aortic Aneury...",2006,yes,yes,yes,"With monitoring of MEP and SSEP, sacrifice--wi...",17062225,"No, reimplantation of segmental (intercostal/l..."
7,Are sports medicine journals relevant and appl...,[To examine the evidence base of sports medici...,"[OBJECTIVE, METHODS, RESULTS]","[Health Personnel, Humans, Periodicals as Topi...",2004,no,no,no,There is a dearth of studies addressing diagno...,15388567,Sports medicine journals are relevant to pract...
8,Transient tachypnea of the newborn (TTN): a ro...,[DNA was collected for genotyping from 73 term...,"[METHODS, RESULTS]","[Apgar Score, Case-Control Studies, DNA, Femal...",2008,yes,yes,yes,We conclude that beta1Gly49 homozygosity and T...,18540901,Transient tachypnea of the newborn (TTN) is th...
9,Hepatorenal syndrome: are we missing some prog...,[Hepatorenal syndrome (HRS) is the functional ...,"[BACKGROUND, METHODS, RESULTS]","[Albumins, Alcoholism, Autoimmune Diseases, Cr...",2012,yes,yes,yes,We report for the first time ESLD etiology as ...,21850494,"Yes, several prognostic factors in hepatorenal..."


In [69]:
final_df.to_csv('./pubmedqa/data/data-with-sonnet-5-5-answers.csv', index = True)